### Initialise workbench and validate using arguments

In [14]:
from five_safes_tes_workbench.workbench import Workbench

In [15]:
wb = Workbench()

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Template registered: 'analysis'


In [ ]:
wb.validate(
    project="Testing",
    tes_base_url="http://localhost:5034",
    minio_sts_endpoint="http://localhost:9000/sts",
    minio_endpoint="http://localhost:9000",
    minio_output_bucket="11343output",
    tres=["DEMO"],
    client_id="Dare-Control-API",
    client_secret="2e60b956-16bc-4dea-8b49-118a8baac5e5",
    username="globaladminuser",
    password="password123",
    keycloak_url="http://localhost:8085/",
)


### Validate settings using a token instead of keycloak login

In [ ]:
wb.validate(
    project="Testing",
    tes_base_url="http://localhost:5034",
    minio_sts_endpoint="http://localhost:9000/sts",
    minio_endpoint="http://localhost:9000",
    minio_output_bucket="11343output",
    tres=["DEMO"],
    access_token="your-token-here",
)

### Validate using config.yml file

In [16]:
wb = Workbench()
wb.validate(config_path="config.yml") #type: ignore

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Template registered: 'analysis'
INFO | Validation successful
INFO | Config: project='NottinghamDemo' tes_base_url='https://api.5s-tes.federated-research.com/' minio_sts_endpoint='https://api.s3.5s-tes.federated-research.com/' minio_endpoint='https://api.s3.5s-tes.federated-research.com/' minio_output_bucket='12590output' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS


### Build a simple example using a template and submit

In [ ]:
# ----- Hello World TES Task Template -----

wb.build_tes.hello_world()

wb.submit()

### Build a TES message from scratch

In [ ]:
# ----- Custom TES Task Template -----

wb.build_tes.custom(
    name="Test Custom mode from Workbench",
    description="cutom analysis",
    executors=[
        {
            "image": "ubuntu",
            "command": ["echo", "Hello World"],
            "workdir": "/outputs",
            "stdout": "/outputs/stdout"
        }
    ],
    outputs=[
        {
            "name": "Stdout",
            "description": "Stdout results",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY"
    }
  ],
)

wb.submit()

### Submit an SQL query using the SQL container with template

In [ ]:
# ----- SQL Query Template -----

query = (
    'WITH user_query AS ('
    'SELECT value_as_number FROM "NottinghamDemo".measurement '
    'WHERE measurement_concept_id = 3000905 '
    'AND value_as_number IS NOT NULL'
    ') SELECT COUNT(*) AS n, SUM(value_as_number) AS total FROM user_query;'
)

wb.build_tes.simple_sql(
    name="Simple SQL Task from Workbench",
    query=query,
)

wb.submit()

### Submit a bunny request using a template

In [ ]:
# ----- Bunny Template -----

wb.build_tes.bunny(
    name="Bunny testing from Workbench",
    command=[
        "--body-json",
        '{"code":"GENERIC","analysis":"DISTRIBUTION","uuid":"123","collection":"test","owner":"me"}',
        "--output",
        "/outputs/output.json",
        "--no-encode",
    ],
)

wb.submit()

### Submit a request to demo analysis container using a template

In [17]:
# ----- Analysis Template -----

query = "SELECT value_as_number FROM \"NottinghamDemo\".measurement \nWHERE measurement_concept_id = 43055141\nAND value_as_number IS NOT NULL"

wb.build_tes.analysis(
    name="Analysis Template testing from Workbench",
    query=query,
    analysis_type="mean",
)

wb.submit()

INFO | Building TES task from template: 'analysis'
INFO | Resolving template: 'analysis'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "Analysis Template testing from Workbench",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Analysis Template testing from Workbench",
         "description": "Analysis Task"
      }
   ],
   "executors": [
      {
         "image": "ghcr.io/health-informatics-uon/five-safes-tes-analytics-dev:sha-dbf029b",
         "command": [
            "--user-query=SELECT value_as_number FROM \"NottinghamDemo\".measurement \nWHERE measurement_concept_id = 43055141\nAND value_as_number IS NOT NULL",
            "--analysis=mean",
            "--output-filename=/outputs/output",
            "--output-format=json"
         ],
         "workdir": "/app",
         "env": {}
      }
   ],
   "volumes": [],
   "tags": {
      "project": "NottinghamDemo",
      "tres": "Nott

'1727'

## S3 Commands

### Fetch all outputs from last submission

In [ ]:
wb.fetch_outputs()


### Fetch last submitted output from named TRE only

In [ ]:

wb.fetch_outputs(tre="Nottingham TRE 01")


### Fetch all outputs by (parent) task ID

In [18]:

wb.fetch_outputs(task_id=1677)


INFO | Fetching ID token from Keycloak for STS...
INFO | Requesting Keycloak tokens from https://drs-core-identity.azurewebsites.net/realms/Dare-Control/protocol/openid-connect/token
INFO | Keycloak tokens fetched successfully
INFO | Exchanging ID token for object-storage credentials via STS (https://api.s3.5s-tes.federated-research.com/)
INFO | MinIO client initialized (endpoint=https://api.s3.5s-tes.federated-research.com/, secure=True)
INFO | Child task info: 1678, TRE: Nottingham TRE 01, status: Completed
INFO | Found 1 result object(s) for task 1678
INFO | Downloading result object: 1678/output.csv
INFO | Downloaded 1678/output.csv -> /home/mszag6/Code/5S-TES-Workbench/src/five_safes_tes_workbench/notebooks/output/Nottingham TRE 01/1678/output.csv
INFO | Child task info: 1679, TRE: Nottingham TRE 02, status: Completed
INFO | Found 1 result object(s) for task 1679
INFO | Downloading result object: 1679/output.csv
INFO | Downloaded 1679/output.csv -> /home/mszag6/Code/5S-TES-Workben

{'Nottingham TRE 01': [PosixPath('/home/mszag6/Code/5S-TES-Workbench/src/five_safes_tes_workbench/notebooks/output/Nottingham TRE 01/1678/output.csv')],
 'Nottingham TRE 02': [PosixPath('/home/mszag6/Code/5S-TES-Workbench/src/five_safes_tes_workbench/notebooks/output/Nottingham TRE 02/1679/output.csv')]}

### Fetch outputs from named TRE with specified parent task ID

In [ ]:

wb.fetch_outputs(task_id=1677, tre="Nottingham TRE 02")